# Semantic web & RDF — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def truth_rows(names):
    return [dict(zip(names, vals)) for vals in itertools.product([False, True], repeat=len(names))]
def draw_graph(nodes, edges, title='graph'):
    angles=np.linspace(0, 2*np.pi, len(nodes), endpoint=False)
    pos={n:(np.cos(a), np.sin(a)) for n,a in zip(nodes, angles)}
    plt.figure(figsize=(5,4))
    for a,b in edges:
        xa,ya=pos[a]; xb,yb=pos[b]; plt.plot([xa,xb],[ya,yb], 'k-', alpha=.6)
    for n,(x,y) in pos.items():
        plt.scatter([x],[y], s=500, zorder=3); plt.text(x,y,n,ha='center',va='center',color='white',weight='bold')
    plt.axis('equal'); plt.axis('off'); plt.title(title)
    return pos
print('setup ok')

## Reasoning over triples as a graph
RDF stores knowledge as subject-predicate-object triples. These examples visualize triples, class membership, subclass closure, a SPARQL-like pattern match, and a small linked-data join.

In [ ]:
# 1) triples are graph edges
triples=[('ex:ada','rdf:type','ex:Professor'),('ex:ada','ex:worksAt','ex:Uni')]
nodes=sorted({s for s,_,o in triples}|{o for s,_,o in triples}); draw_graph(nodes, [(s,o) for s,_,o in triples], 'RDF triples as edges')
assert len(triples)==2

In [ ]:
# 2) rdf:type answers class membership queries
triples=[('ada','type','Professor'),('bo','type','Student'),('cy','type','Professor')]
profs=[s for s,p,o in triples if p=='type' and o=='Professor']
plt.figure(figsize=(5,3)); plt.bar(['ada','bo','cy'], [x in profs for x in ['ada','bo','cy']], color='tab:blue'); plt.title('type Professor query')
assert profs==['ada','cy']

In [ ]:
# 3) rdfs:subClassOf closure adds inherited types
sub={('Professor','Person')}; inferred={(s,'type',sup) for s,p,c in triples for c2,sup in sub if p=='type' and c==c2}
plt.figure(figsize=(5,3)); plt.bar(['explicit','inferred'], [len(triples),len(inferred)], color=['tab:gray','tab:green']); plt.title('subclass closure')
assert ('ada','type','Person') in inferred and len(inferred)==2

In [ ]:
# 4) SPARQL-like pattern: ?x worksAt Uni
triples=[('ada','worksAt','Uni'),('bo','worksAt','Lab'),('cy','worksAt','Uni')]; ans=[s for s,p,o in triples if p=='worksAt' and o=='Uni']
plt.figure(figsize=(5,3)); plt.bar(['ada','bo','cy'], [x in ans for x in ['ada','bo','cy']], color='tab:purple'); plt.title('pattern match ?x worksAt Uni')
assert ans==['ada','cy']

In [ ]:
# 5) linked-data join: type plus workplace
T=[('ada','type','Professor'),('bo','type','Student'),('cy','type','Professor'),('ada','worksAt','Uni'),('cy','worksAt','Uni')]
ans=[s for s,_,o in T if o=='Professor' and (s,'worksAt','Uni') in T]
plt.figure(figsize=(5,3)); plt.bar(ans,[1]*len(ans),color='tab:green'); plt.title('professors at Uni')
assert ans==['ada','cy']